# Capstone 1 - Machine Learning

* Student: Oscar Juárez
* Student ID: 2026801
* Higher Diploma on Data Analysis for Business
* Machine Learning
* November 2026

## Indian Liver Patient Dataset

Death by liver cirrhosis continues to increase, given the increase in alcohol consumption rates, chronic hepatitis infections, and obesity-related liver disease. Notwithstanding the high mortality of this disease, liver diseases do not affect all sub-populations equally. The early detection of pathology is a determinant of patient outcomes, yet female patients appear to be marginalized when it comes to early diagnosis of liver pathology. The dataset comprises 584 patient records collected from the NorthEast of Andhra Pradesh, India. The prediction task is to determine whether a patient suffers from liver disease based on the information about several biochemical markers, including albumin and other enzymes required for metabolism.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split             # Import train_test_split function
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn import metrics                                      # Import scikit-learn metrics module for accuracy calculation
from sklearn import tree

## Exploring the Dataset

The dataset itself doesn't include any column names, to which we are going to use the ones described at [UCI's Site](https://archive.ics.uci.edu/dataset/225/ilpd+indian+liver+patient+dataset).

### Variables Table

| Variable Name | Role | Type | Demographic | Description | Units | Missing Values |
|---|---|---|---|---|---|---|
| Age | Feature | Integer | Age | Age of the patient. Any patient whose age exceeded 89 is listed as being of age "90". | years | no |
| Gender | Feature | Binary | Gender | Gender of the patient |  | no |
| TB | Feature | Continuous |  | Total Bilirubin |  | no |
| DB | Feature | Continuous |  | Direct Bilirubin |  | no |
| Alkphos | Feature | Integer |  | Alkaline Phosphotase |  | no |
| Sgpt | Feature | Integer |  | Alamine Aminotransferase |  | no |
| Sgot | Feature | Integer |  | Aspartate Aminotransferase |  | no |
| TP | Feature | Continuous |  | Total Proteins |  | no |
| ALB | Feature | Continuous |  | Albumin |  | no |
| A/G Ratio | Feature | Continuous |  | Albumin and Globulin Ratio |  | no |
| Selector | Target | Binary |  | Selector field used to split the data into two sets (labeled by the experts) |  | no |

In [2]:
COLUMNS = ["Age", "Gender", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/G Ratio", "Selector"]
df = pd.read_csv(
  "indian-liver-patient-dataset.csv",
  header=None,
  names=COLUMNS
)
df.head()

,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        583 non-null    int64  
 1   Gender     583 non-null    str    
 2   TB         583 non-null    float64
 3   DB         583 non-null    float64
 4   Alkphos    583 non-null    int64  
 5   Sgpt       583 non-null    int64  
 6   Sgot       583 non-null    int64  
 7   TP         583 non-null    float64
 8   ALB        583 non-null    float64
 9   A/G Ratio  579 non-null    float64
 10  Selector   583 non-null    int64  
dtypes: float64(5), int64(5), str(1)
memory usage: 52.8 KB


The dataset doesn't have any null values, except for the A/G Ratio column with 4 missing values which we are going to drop since it's a small quantity of rows.

In [4]:
df.dropna(inplace=True)
df.isnull().sum()

Age          0
Gender       0
TB           0
DB           0
Alkphos      0
Sgpt         0
Sgot         0
TP           0
ALB          0
A/G Ratio    0
Selector     0
dtype: int64

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,579.0,44.782383,16.221786,4.0,33.0,45.00,58.0,90.0
TB,579.0,3.315371,6.227716,0.4,0.8,1.00,2.6,75.0
DB,579.0,1.494128,2.816499,0.1,0.2,0.30,1.3,19.7
Alkphos,579.0,291.366149,243.561863,63.0,175.5,208.00,298.0,2110.0
Sgpt,579.0,81.126079,183.182845,10.0,23.0,35.00,61.0,2000.0
Sgot,579.0,110.414508,289.850034,10.0,25.0,42.00,87.0,4929.0
TP,579.0,6.481693,1.084641,2.7,5.8,6.60,7.2,9.6
ALB,579.0,3.138515,0.794435,0.9,2.6,3.10,3.8,5.5
A/G Ratio,579.0,0.947064,0.319592,0.3,0.7,0.93,1.1,2.8
Selector,579.0,1.284974,0.451792,1.0,1.0,1.00,2.0,2.0


How many patients from the dataset have liver disease?

In [6]:
df["Selector"].value_counts(normalize=True).round(3)

Selector
1    0.715
2    0.285
Name: proportion, dtype: float64

Let's see if the gender count is balanced.

In [7]:
gender_counts = df["Gender"].value_counts()
gender_counts

Gender
Male      439
Female    140
Name: count, dtype: int64

### Key Findings

* **Sgot** has a positive skew, since the mean is 110.41 and has a median of 42.0
* 71.5% of patients have been diagnosed with a liver disease (baseline).
* The gender columns is not balanced, therefore a SMOTE should be applied.

## Training and testing the models

For the model training, we are going to split the data in 75/25, 85/15 and 95/5. We are going to use all the columns provided by the dataset, except the selector column.

Separating target column from other features to separate X and y values

In [ ]:
X = df.iloc[:,:-1] # All the columns except the last one
y = df["Selector"] # Only selector column

print(X, y)

     Age  Gender    TB   DB  Alkphos  Sgpt  Sgot   TP  ALB  A/G Ratio
0     65       0   0.7  0.1      187    16    18  6.8  3.3       0.90
1     62       1  10.9  5.5      699    64   100  7.5  3.2       0.74
2     62       1   7.3  4.1      490    60    68  7.0  3.3       0.89
3     58       1   1.0  0.4      182    14    20  6.8  3.4       1.00
4     72       1   3.9  2.0      195    27    59  7.3  2.4       0.40
..   ...     ...   ...  ...      ...   ...   ...  ...  ...        ...
578   60       1   0.5  0.1      500    20    34  5.9  1.6       0.37
579   40       1   0.6  0.1       98    35    31  6.0  3.2       1.10
580   52       1   0.8  0.2      245    48    49  6.4  3.2       1.00
581   31       1   1.3  0.5      184    29    32  6.8  3.4       1.00
582   38       1   1.0  0.3      216    21    24  7.3  4.4       1.50

[579 rows x 10 columns] 0      1
1      1
2      1
3      1
4      1
      ..
578    2
579    1
580    1
581    1
582    2
Name: Selector, Length: 579, dtype: 

### Random Forest

It splits on thresholds, so skew and outliers matter little and it needs no scaling. Scikit-learn's Random Forest only accepts numbers, so the gender column should be encoded. We going to encode it so that:

* Female = 0
* Male = 1

In [8]:
le = LabelEncoder()

df["Gender"] = le.fit_transform(df["Gender"])
df["Gender"].head()

0    0
1    1
2    1
3    1
4    1
Name: Gender, dtype: int64

#### ¿Apply SMOTE to balance the data?

Since there is an imbalance on the gender columns, **we shoud apply SMOTE in order to balance the trainning set**. The smote will be specified by passing a boolean variable to the get_model_performance function.

#### Creating a model performance function

This function will allow us to define which model we want to get the performance from, if smote should apply and the trainning splits. By default, smote is False and the trainning splits are defined as 75/25, 85/15, 95/5.

In [10]:
def get_model_performance(model, apply_smote=False, trainning_splits=[0.25, 0.15, 0.05]):
  smote = SMOTE(random_state = 777)
  results = []
  for split in trainning_splits:
    # Split the data into trainning and testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = split, random_state = 777)
    # Apply smote to the tranning data only if specified

    if (apply_smote):
      X_train, y_train = smote.fit_resample(X_train, y_train)

    # Train the model
    model.fit(X_train, y_train)
    # Predict
    y_pred = clf_RF.predict(X_test)
    # Get the model's accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Get the classification report with each value
    report = classification_report(y_test, y_pred, output_dict=True)
    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']
    f1_score = report['weighted avg']['f1-score']

    results.append([(1 - split), accuracy, precision, recall, f1_score])

  # Get the cross validation scores
  cv_scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')

  return results, cv_scores


#### Getting Random Forest Performance

We are going to get the performance for all the splits in the random forest model. 

In [11]:
clf_RF = RandomForestClassifier(n_estimators = 1000, random_state = 777)
random_forest_performance, random_forest_cv = get_model_performance(clf_RF, True)
random_forest_df = pd.DataFrame(random_forest_performance, columns=["Split", "Accuracy", "Precision", "Recall", "F1 Score"])
print(f"""
Random Forest Performance:
{random_forest_df.mul(100).sort_values(by="Precision", ascending=False)}

Mean CV Accuracy:
{(random_forest_cv.mean() * 100):.3f}%
""")


Random Forest Performance:
   Split   Accuracy  Precision     Recall   F1 Score
1   85.0  71.264368  72.144314  71.264368  71.620122
0   75.0  68.965517  69.195881  68.965517  69.077509
2   95.0  62.068966  59.449172  62.068966  60.109830

Mean CV Accuracy:
70.130%



#### Findings





For the 75/25 split, the test accuracy has a result of 69%, which aligns with its cross validation mean of 69.4%. At 85/15, we have a result of 69% test accuracy and the cross validation mean is 69.5% so the two measures agree at both splits. On the other hand, the 95/5 split has a higher cross validation mean than the test accuracy, at 72.4% and 58.6% respectively; this performance happens due to the small test set: with only 29 test rows to evaluate the model, 17 out of 29 were correct but it doesn't mean it reflects the accuracy correctly, therefore a 58.6% accuracy is less reliable and 72.4% should be trusted because of the amount of cross validation tests. With a baseline of 71.5% and the highest accuracy of 72.4% with the 95 split, the random forest model barely any better to classify liver disease than to assume everyone has a liver disease.

---

### SVM

It splits on thresholds, so skew and outliers matter little and it needs no scaling. Scikit-learn's Random Forest only accepts numbers, so the gender column should be encoded. We going to encode it so that:

* Female = 0
* Male = 1